<a href="https://colab.research.google.com/github/IsaacGSolis/Mineria-de-Datos-IEGS/blob/main/BayesGaussianNB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

**Parte 1: Carga y limpieza (identificar columnas númericas)**

In [8]:
df=pd.read_csv('/content/credit_risk_dataset.csv')

df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [10]:
df.isnull().sum()

,0
person_age,0
person_income,0
person_home_ownership,0
person_emp_length,895
loan_intent,0
loan_grade,0
loan_amnt,0
loan_int_rate,3116
loan_status,0
loan_percent_income,0


In [11]:
df=df.dropna()

In [13]:
col_cat=df.select_dtypes(include=['object']).columns
col_cat

Index(['person_home_ownership', 'loan_intent', 'loan_grade',
       'cb_person_default_on_file'],
      dtype='object')

In [14]:
le_dict={}
for col in col_cat:
    le=LabelEncoder()
    df[col]=le.fit_transform(df[col])
    le_dict[col]=le

In [15]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,3,123.0,4,3,35000,16.02,1,0.59,1,3
1,21,9600,2,5.0,1,1,1000,11.14,0,0.10,0,2
2,25,9600,0,1.0,3,2,5500,12.87,1,0.57,0,3
3,23,65500,3,4.0,3,2,35000,15.23,1,0.53,0,2
4,24,54400,3,8.0,3,2,35000,14.27,1,0.55,1,4


**Parte 2: Exploración**

In [16]:
df.describe()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
count,28638.000000,2.863800e+04,28638.000000,28638.000000,28638.000000,28638.000000,28638.000000,28638.000000,28638.000000,28638.000000,28638.000000,28638.000000
mean,27.727216,6.664937e+04,1.680669,4.788672,2.531322,1.228158,9656.493121,11.039867,0.216600,0.169488,0.178190,5.793736
std,6.310441,6.235645e+04,1.434497,4.154627,1.729816,1.170746,6329.683361,3.229372,0.411935,0.106393,0.382679,4.038483
min,20.000000,4.000000e+03,0.000000,0.000000,0.000000,0.000000,500.000000,5.420000,0.000000,0.000000,0.000000,2.000000
25%,23.000000,3.948000e+04,0.000000,2.000000,1.000000,0.000000,5000.000000,7.900000,0.000000,0.090000,0.000000,3.000000
50%,26.000000,5.595600e+04,3.000000,4.000000,3.000000,1.000000,8000.000000,10.990000,0.000000,0.150000,0.000000,4.000000
75%,30.000000,8.000000e+04,3.000000,7.000000,4.000000,2.000000,12500.000000,13.480000,0.000000,0.230000,0.000000,8.000000
max,144.000000,6.000000e+06,3.000000,123.000000,5.000000,6.000000,35000.000000,23.220000,1.000000,0.830000,1.000000,30.000000


In [30]:
df.corr

<bound method DataFrame.corr of        person_age  person_income  person_home_ownership  person_emp_length  \
0              22          59000                      3              123.0   
1              21           9600                      2                5.0   
2              25           9600                      0                1.0   
3              23          65500                      3                4.0   
4              24          54400                      3                8.0   
...           ...            ...                    ...                ...   
32576          57          53000                      0                1.0   
32577          54         120000                      0                4.0   
32578          65          76000                      3                3.0   
32579          56         150000                      0                5.0   
32580          66          42000                      3                2.0   

       loan_intent  loan_grade  loan_amnt  loan_int_rate  loan_status  \
0                4           3      35000          16.02            1   
1                1           1       1000          11.14            0   
2                3           2       5500          12.87            1   
3                3           2      35000          15.23            1   
4                3           2      35000          14.27            1   
...            ...         ...        ...            ...          ...   
32576            4           2       5800          13.16            0   
32577            4           0      17625           7.49            0   
32578            2           1      35000          10.99            1   
32579            4           1      15000          11.48            0   
32580            3           1       6475           9.99            0   

       loan_percent_income  cb_person_default_on_file  \
0                     0.59                          1   
1                     0.10                          0   
2                     0.57                          0   
3                     0.53                          0   
4                     0.55                          1   
...                    ...                        ...   
32576                 0.11                          0   
32577                 0.15                          0   
32578                 0.46                          0   
32579                 0.10                          0   
32580                 0.15                          0   

       cb_person_cred_hist_length  
0                               3  
1                               2  
2                               3  
3                               2  
4                               4  
...                           ...  
32576                          30  
32577                          19  
32578                          28  
32579                          26  
32580                          30  

[28638 rows x 12 columns]>

**Parte 3: Identificación de variables X, Y**

In [18]:
X=df.drop('loan_status', axis=1)
y=df['loan_status']

**Parte 4: División train, test**

In [19]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

**Parte 5: Modelo GaussianNB**

In [20]:
modelo=GaussianNB()
modelo.fit(X_train,y_train)

GaussianNB()

**Parte 6: Predicción**

In [21]:
y_pred=modelo.predict(X_test)
y_pred

array([0, 1, 0, ..., 0, 0, 0])

**Parte 7: Evaluación**

In [26]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))



Accuracy: 0.8171554934823091


In [27]:
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))


Matriz de confusión:
[[6280  435]
 [1136  741]]


In [28]:
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))


Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.85      0.94      0.89      6715
           1       0.63      0.39      0.49      1877

    accuracy                           0.82      8592
   macro avg       0.74      0.66      0.69      8592
weighted avg       0.80      0.82      0.80      8592



**Parte 8: Reflexión**

**¿Por qué GaussianNB funciona aquí?**

Funciona aqui porque el Data set tiene variables que siguen una distribucion normal o gaussiana, aparte de que contiene variables numericas y este tipo de modelo funciona muy bien con variables numericas.

**¿Qué variable crees que más influye?**

Las variables con mayor influencia en el dataset seria el amount o monto, que al tratarse de datos que describen el pretamo crediticio, es una variable que podria considerarse principal, despues serian los ingresos de la persona, para relacionar los riesgos del prestamos.